In [2]:
import requests
import configparser
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
from utils import plot_image, get_access_token
import numpy as np
from rasterio.io import MemoryFile
from datetime import datetime, timedelta
from sentinelhub import BBox, bbox_to_dimensions, CRS
from sentinelhub import SHConfig, SentinelHubCatalog, DataCollection
import evalscripts as eval
import pandas as pd
import os
import rasterio

In [4]:
config_file = configparser.ConfigParser()
config_file.read("config.ini")

username = config_file["copernicus"]["username"]
password = config_file["copernicus"]["password"]

config = SHConfig()
config.sh_client_id = config_file["copernicus"]["client_id"] #"<CLIENT ID>"
config.sh_client_secret = config_file["copernicus"]["client_secret"] #<CLIENT SECRET>"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token" # Is it required?
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.save("cdse")
config = SHConfig("cdse")

In [ ]:
def download_image_copernicus(access_token, time_interval, image_type, aoi, evalscript, resolution, config, save_path, cloud_cover_limit): 

    # Set the url to sent the request
    url = "https://sh.dataspace.copernicus.eu/api/v1/process"

    # Define the Area of Interest
    aoi_bbox = BBox(bbox=aoi, crs=CRS.WGS84)
    aoi_size = bbox_to_dimensions(aoi_bbox, resolution=resolution)

    # Use SentinelHubCatalog to check if there are any images in the time interval of interest
    # If there are, extract the date (can't get it directly from the tiff)
    catalog = SentinelHubCatalog(config=config)
    date_range = time_interval[0],  time_interval[1]
    search_iterator = catalog.search(
        DataCollection.SENTINEL2_L2A,
        bbox=aoi_bbox,
        time= date_range ,
        fields={"include": ["id", "properties.eo:cloud_cover"], "exclude": [ "properties.datetime"]},
    )
    results = list(search_iterator)
    unique_results = {}
    # A partir del id guardamos las fechas
    for item in results:
        acquisition_id = item['id'].split('_T')[0]
        if acquisition_id not in unique_results:
            unique_results[acquisition_id] = item
    unique_results = list(unique_results.values())
    ids = [item['id'] for item in unique_results]
    dates = [datetime.strptime(id.split('_')[2][:8], "%Y%m%d").date() for id in ids]
    
    # Guardamos cloud cover de las imágenes en el intervalo
    cloud_covers = []
    for item in results:
        cloud_cover = item["properties"]["eo:cloud_cover"]
        cloud_covers.append(cloud_cover)

    if len(dates) == 0: 
        print(f"Request empty for dates {time_interval}")
        return None, None
    
    # Comprobar si la imagen está cubierta por nubes antes de seguir
    elif any(cc > cloud_cover_limit for cc in cloud_covers):
        print(f"Request covered by clouds for dates {time_interval}, with coverage {cloud_covers}")
        return None, None       

    else: 
        date_string = [d.isoformat() for d in dates][0]

        # Define the content for the post
        headers={
        "Content-Type": "application/json",
        "Authorization" : "Bearer "+ access_token
        }


        json={
            "input": {
                "bounds": {
                    "bbox": aoi
                },
                "data": [
                    {
                    "dataFilter": {
                        "timeRange": {
                        "from": time_interval[0] + "T00:00:00Z", #"2024-10-12T00:00:00Z", #YYYY-mm-dd
                        "to": time_interval[1] + "T23:59:59Z"#"2024-11-12T23:59:59Z"
                        },
                        "mosaickingOrder": "leastCC"
                    
                    },
                    "type": "sentinel-2-l2a"
                    }]
            },
            "output": {
                "width": aoi_size[0],#1271,
                "height": aoi_size[1],#2183

                "responses": [
                    {
                        "format": {
                            "type": "image/" + image_type
                        }
                    }
                ]
            },
            "evalscript" : evalscript,
            "data_folder" : "test_dir",
            "save_data" : True
        }

        response = requests.post(url, headers=headers, json=json)

        if response.status_code == 200:
            print(f"Request completed for date {date_string} in interval {time_interval}")
            
            if save_path:
                file_name = f"{date_string}.tiff"
                file_path = os.path.join(save_path, file_name)

                with MemoryFile(response.content) as memfile:
                    with memfile.open() as dataset:
                        with rasterio.open(file_path, 'w', **dataset.profile) as dst:
                            dst.write(dataset.read())
                print(f"Saved TIFF file to {file_path}")
            
            return response, date_string
        else:
            print(f"Request failed {response.content}")
            return None, None

In [ ]:
evalscript_all_bands = """
    //VERSION=3
    function setup() {
        return {
            input: [{
                bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12","SCL"]
            }],
            output: {
                bands: 13
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B01,
                sample.B02,
                sample.B03,
                sample.B04,
                sample.B05,
                sample.B06,
                sample.B07,
                sample.B08,
                sample.B8A,
                sample.B09,
                sample.B11,
                sample.B12,
                sample.SCL];
    }
"""

In [ ]:
access_token = get_access_token(username, password)
evalscript = evalscript_all_bands
image_type = "tiff" # no se usa
aoi = [-0.866977, 37.628916, -0.71696, 37.822802]
resolution = 10
cloud_cover_limit = 75
## Time interval ## 
year = 2021
start = datetime(year, 1, 1)
end = datetime(year, 2, 10)
tdelta = timedelta(days=5)
n_chunks = round((end - start)/tdelta)
starts = [(start + i * tdelta).date().isoformat() for i in range(n_chunks+1)]
ends = [(start + timedelta(days=4) + i * tdelta).date().isoformat() for i in range(n_chunks+1)]
slots = [(starts[i], ends[i]) for i in range(len(starts))]
folder = "test_dir"

In [ ]:
for time_interval in slots:
    # Get the satellite response for the current time interval
    response, date_taken = download_image_copernicus(access_token, time_interval, image_type, aoi, evalscript, resolution, config, folder, cloud_cover_limit)
    if response is None:
        print(f"No data available for time interval {time_interval}")
        continue 

    if response.status_code == 200:
        print("Ok")